# Chapter 8 — Mastering the Model Context Protocol (MCP)

**Multi-Agent Analog EDA — PhD-Level Notebook**

---

The **Model Context Protocol (MCP)** is a *standardized* contract between **language-model hosts** and **external capabilities** (simulators, PDK databases, layout engines, revision control, and proprietary EDA APIs).
Informally, MCP is often called the **“USB‑C of AI”**: one mechanical and electrical interface replaced dozens of charger shapes; similarly, MCP aims to replace *ad hoc* agent↔tool wiring with **negotiable capabilities**, **typed schemas**, and **lifecycle-aware transports**.

This chapter develops MCP from first principles for **analog EDA**: we dissect the **host–client–server** topology, relate it to **JSON‑RPC 2.0**, implement a **pedagogical Ngspice MCP server** with **FastAPI** (HTTP JSON‑RPC surface;
stdio/SSE are conceptually identical at the message layer), and show how **agents** discover tools, validate arguments, and orchestrate **multi‑server** graphs (Ngspice + ALIGN + OpenROAD).

### Learning objectives

1. **Decompose** MCP into **Host**, **Client**, and **Server** roles and map them onto Claude Desktop / IDE plugins / custom orchestrators.
2. **Classify** MCP capabilities into **Tools**, **Resources**, and **Prompts** and connect each to analog workflow artifacts (netlists, corners, DRC decks, debug heuristics).
3. **Explain** **JSON‑RPC 2.0** framing, **capability negotiation** at `initialize`, and clean **shutdown** semantics for long‑running simulation farms.
4. **Implement** a **functional** Ngspice‑oriented MCP server (mocked kernel) exposing `run_simulation`, `get_device_parameters`, `extract_metrics`.
5. **Engineer** an **MCP client** for agents: **schema validation**, **retries**, **idempotency keys**, and **connection lifecycle** management.
6. **Compose** **advanced patterns**: federated tool namespaces across servers, **resource subscriptions** for simulation progress, and **domain prompts** for analog reasoning.

### Notation

- Host application $\mathcal{H}$ runs model $\mathcal{M}$ and one or more MCP **clients** $\mathcal{C}_i$.
- Server $S_j$ advertises capability multiset $\mathcal{K}_j = \mathcal{T}_j \cup \mathcal{R}_j \cup \mathcal{P}_j$ (tools, resources, prompts).
- JSON‑RPC message $m = (\text{method}, \theta, \text{id})$ with parameters $\theta$ validated against JSON Schema (conceptually $\theta \in \Theta_{\text{valid}}$).

---


## 8.1 MCP architecture deep dive

### 8.1.1 Roles

| Role | Responsibility | Analog EDA example |
|------|----------------|----------------------|
| **MCP Host** | Owns the user session, UI, model routing, policy (what the LLM may do). | Jupyter agent kernel, Claude Desktop, custom VS Code extension launching a sizing copilot. |
| **MCP Client** | Maintains transport to **one** server, tracks protocol state, marshals JSON‑RPC. | Python/async task inside the host that speaks HTTP/SSE/stdio to a simulator microservice. |
| **MCP Server** | Exposes **Tools**, **Resources**, **Prompts** with machine‑readable metadata. | Ngspice wrapper, PDK file server, ALIGN/OpenROAD driver, encrypted waveform store. |

### 8.1.2 Capability classes

1. **Tools** — *Imperative* operations with typed arguments (side‑effects allowed): run SPICE, push layout edits, launch LVS.
2. **Resources** — *Declarative* addressable artifacts (read‑mostly): model cards, constraint JSON, previous simulation logs (`uri` + MIME).
3. **Prompts** — *Templated* reasoning packages combining system + user slots: “debug ringing on node `v_out` given this netlist fragment”.

### 8.1.3 JSON‑RPC 2.0 as the lingua franca

MCP messages are **JSON objects** with `jsonrpc: "2.0"`, a `method` string, optional `params`, and optional `id`. **Requests** carry `id`; **notifications** omit it. **Responses** return either `result` or `error` with the same `id`.

### 8.1.4 Lifecycle (conceptual state machine)

1. **Transport connect** (stdio pipe, SSE channel, HTTP keep‑alive).
2. **`initialize`** — exchange protocol versions and capability flags (tools? resources? prompts? sampling? roots?).
3. **Capability discovery** — `tools/list`, `resources/list`, `prompts/list` (may be cached with ETags / revision tokens).
4. **Operation** — `tools/call`, `resources/read`, `prompts/get`, optional **subscriptions** / progress notifications.
5. **Shutdown** — cancel in‑flight tasks, flush logs, release file locks on PDK views.

The next cell renders a **GitHub‑dark** block diagram of the topology.

---


In [ ]:
import sys; sys.path.insert(0, '..')
from style_utils import (setup_3b1b_style, glow_line, glow_fill, styled_box,
                         styled_arrow, finish_plot, plotly_3b1b_layout,
                         BACKGROUND, SURFACE, TEXT, TEXT_DIM, GRID,
                         BLUE, TEAL, GREEN, YELLOW, GOLD, RED,
                         ROSE, PURPLE, CYAN, ORANGE, PALETTE)
setup_3b1b_style()

# Global imports, GitHub-dark matplotlib + plotly_dark
from __future__ import annotations

import json
import textwrap
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

DARK_BG = "#0d1117"
PANEL = "#161b22"
ACCENT = "#58a6ff"
GREEN = "#3fb950"
ORANGE = "#d29922"
PURPLE = "#bc8cff"
RED = "#f85149"

mpl.rcParams.update(
    {
        "figure.facecolor": DARK_BG,
        "axes.facecolor": DARK_BG,
        "axes.edgecolor": "#30363d",
        "axes.labelcolor": "#c9d1d9",
        "text.color": "#c9d1d9",
        "xtick.color": "#8b949e",
        "ytick.color": "#8b949e",
        "font.size": 11,
    }
)
pio.templates.default = "plotly_dark"

fig, ax = plt.subplots(figsize=(11, 6), dpi=120)
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis("off")
ax.set_title("MCP topology: Host → Clients → Servers (Tools / Resources / Prompts)", color="#f0f6fc", fontsize=13, pad=12)


def box(xy, w, h, label, sub="", color=ACCENT):
    x, y = xy
    fb = FancyBboxPatch(
        (x, y), w, h, boxstyle="round,pad=0.03,rounding_size=0.15",
        linewidth=1.4, edgecolor=color, facecolor=PANEL,
    )
    ax.add_patch(fb)
    ax.text(x + w / 2, y + h * 0.62, label, ha="center", va="center", color="#f0f6fc", fontsize=11, fontweight="600")
    if sub:
        ax.text(x + w / 2, y + h * 0.28, sub, ha="center", va="center", color="#8b949e", fontsize=9)


def arrow(p1, p2, color="#8b949e"):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle="-|>", mutation_scale=14, linewidth=1.2, color=color))


# Host
box((0.4, 4.6), 2.6, 1.6, "MCP Host", "UI + policy + model router", ACCENT)
ax.text(1.7, 6.35, "LLM", ha="center", color=PURPLE, fontsize=10, fontstyle="italic")

# Clients
box((3.6, 5.0), 2.2, 1.1, "MCP Client A", "JSON-RPC session", GREEN)
box((3.6, 3.5), 2.2, 1.1, "MCP Client B", "JSON-RPC session", GREEN)

# Servers
box((6.5, 5.0), 2.9, 1.1, "Server: Ngspice", "tools + resources", ORANGE)
box((6.5, 3.5), 2.9, 1.1, "Server: ALIGN / OpenROAD", "tools + prompts", ORANGE)

# Capability strip
box((1.0, 1.0), 8.0, 1.2, "Advertised capabilities", "Tools · Resources · Prompts (schemas + URIs)", PURPLE)

arrow((3.0, 5.45), (3.6, 5.55))
arrow((3.0, 5.15), (3.6, 3.95))
arrow((5.8, 5.55), (6.5, 5.55))
arrow((5.8, 4.05), (6.5, 4.05))
arrow((5.0, 3.5), (5.0, 2.25), color=ACCENT)
ax.text(5.25, 2.85, "list / call / read / get", color=ACCENT, fontsize=9)

handles = [
    mpatches.Patch(facecolor=PANEL, edgecolor=ACCENT, label="Host"),
    mpatches.Patch(facecolor=PANEL, edgecolor=GREEN, label="Client"),
    mpatches.Patch(facecolor=PANEL, edgecolor=ORANGE, label="Server"),
]
ax.legend(handles=handles, loc="lower right", frameon=True, facecolor=PANEL, edgecolor="#30363d")
plt.tight_layout()
plt.show()

print("Themes: matplotlib face #0d1117, plotly template plotly_dark.")


### 8.1.5 Protocol sketch (JSON‑RPC methods)

The *real* MCP specification evolves; for EDA integration the **important invariant** is: **typed discovery + idempotent framing**. A minimal method set useful for analog agents:

| Phase | Example methods | Purpose |
|-------|-----------------|--------|
| Init | `initialize`, `initialized` (notification) | Version handshake, capability bits |
| Tools | `tools/list`, `tools/call` | Invoke `run_simulation`, etc. |
| Resources | `resources/list`, `resources/read` | Fetch PDK fragments, templates |
| Prompts | `prompts/list`, `prompts/get` | Pull structured reasoning templates |
| Shutdown | `$/cancelRequest`, transport close | Stop long corners, release locks |

```mermaid
flowchart LR
  H[MCP Host] --> C[MCP Client]
  C -->|JSON-RPC 2.0| S[MCP Server]
  S --> T[Tools]
  S --> R[Resources]
  S --> P[Prompts]
```

---


## 8.2 Why MCP for analog EDA?

1. **Eliminates bespoke glue** — Before MCP‑style contracts, every agent stack re‑implemented *ad hoc* HTTP routes, CLI parsers, and LangChain tool wrappers for SPICE. A **single schema** means your **Ngspice** tool looks identical to the model whether the host is Claude, GPT, or an open‑weight local model.

2. **Standardized invocation** — `tools/call` carries **JSON arguments** validated against **JSON Schema** (or Pydantic models server‑side). This is the analog of **strongly typed APIs** in OpenAPI—critical when a typo in `tran 1n 10u` vs `tran 1n 10m` costs hours of CPU.

3. **Transport‑agnostic** — The same message shapes ride over **stdio** (local tools), **SSE** (streaming logs), or **HTTP** (remote EDA farm). Your **security perimeter** moves with the transport: stdio inherits OS user ACLs; HTTP gets mTLS and OAuth2.

4. **Security model for sensitive data** — PDKs under NDA, encrypted **GDSII**, and **foundry rule decks** can remain on a **server** with **scoped credentials**. The host sees *capability metadata*, not raw filesystem paths, and policy engines can **allow‑list** methods per project.

The next cell visualizes **integration surface area**: MCP reduces the number of distinct integration protocols agents must speak (pedagogical, not empirical benchmark).

---


In [ ]:
# Pedagogical: protocol fan-in (fewer host-side adapters with MCP-style uniformity)
labels = ["Raw CLI\nwrappers", "REST\n(one-off)", "gRPC\n(internal)", "MCP-shaped\nJSON-RPC"]
# Toy complexity scores (relative)
before = np.array([9, 7, 6, 2])
x = np.arange(len(labels))

fig = go.Figure(
    go.Bar(
        x=labels,
        y=before,
        marker_color=[RED, ORANGE, PURPLE, GREEN],
        text=[f"{v}" for v in before],
        textposition="outside",
    )
)
fig.update_layout(
    title="Toy: integration complexity per coupling style (lower is better for agents)",
    yaxis_title="Relative glue / schema drift cost",
    paper_bgcolor=DARK_BG,
    plot_bgcolor=PANEL,
    font=dict(color="#c9d1d9"),
    yaxis=dict(gridcolor="#21262d"),
    margin=dict(t=60, b=40),
    height=420,
)
fig.show()

print("Interpretation: unifying on MCP-like discovery reduces host-specific adapter proliferation.")


## 8.3 Implementing an Ngspice MCP server (FastAPI + JSON‑RPC)

We implement a **self-contained** MCP **subset** over HTTP POST `/rpc`. The **simulation kernel is mocked** (deterministic linear dynamics) so no Ngspice binary or licenses are required; swapping the mock for `subprocess.run(["ngspice"], ...)` is isolated in `_mock_spice`.

**Tools**

| Tool | Role |
|------|------|
| `run_simulation` | Accept SPICE deck snippet + analysis mode; returns `run_id` + preview metrics |
| `get_device_parameters` | Return W/L/AD/AS/multiplier for an instance (PDK overlay) |
| `extract_metrics` | Given `run_id`, compute unity-gain bandwidth, DC gain (toy) |

**Resources**

| URI | Content |
|-----|---------|
| `mcp://pdk/models` | BSIM/CMC model card fragment (synthetic) |
| `mcp://templates/ota_cs` | OTA + current-source testbench template |

**Prompts**

| Name | Intent |
|------|--------|
| `analog_design_prompt` | Structured sizing + margin checklist |
| `debug_prompt` | Oscillation / ringing root-cause tree |

---


In [ ]:
# --- Full pedagogical MCP server (FastAPI) ---
import json
import uuid

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import Any, Dict, List, Literal, Optional, Union

JsonRpcId = Union[str, int, None]


class JsonRpcError(BaseModel):
    code: int
    message: str
    data: Optional[Any] = None


class JsonRpcReq(BaseModel):
    jsonrpc: Literal["2.0"]
    method: str
    params: Optional[Dict[str, Any]] = None
    id: Optional[JsonRpcId] = None


def _err(code: int, message: str, data: Any = None) -> Dict[str, Any]:
    return {"code": code, "message": message, "data": data}


class NgspiceMcpState:
    # In-memory run store + subscription channels (toy).

    def __init__(self) -> None:
        self.runs: Dict[str, Dict[str, Any]] = {}
        self.subscribers: Dict[str, List[Dict[str, Any]]] = {}

    def publish(self, run_id: str, event: Dict[str, Any]) -> None:
        for rec in self.subscribers.get(run_id, []):
            rec["events"].append(event)


STATE = NgspiceMcpState()


def _mock_spice(netlist: str, analysis: str, stop_time: float) -> Dict[str, Any]:
    # Deterministic pseudo-SPICE: RC step on node 'out'.
    rng = np.random.default_rng(abs(hash(netlist)) % (2**32))
    t = np.linspace(0, stop_time, 800)
    tau = 1e-9 * (1.0 + 0.1 * (len(netlist) % 7))
    y = 1.0 - np.exp(-t / tau) + 0.002 * rng.standard_normal(t.shape)
    v90 = float(t[np.searchsorted(y, 0.9 * y[-1])])
    return {"t": t.tolist(), "v_out": y.tolist(), "tau_s": tau, "t_90": v90}


def build_app() -> FastAPI:
    app = FastAPI(title="Ngspice MCP (pedagogical)", version="0.1.0")

    TOOL_DEFS = {
        "run_simulation": {
            "description": "Execute (mock) Ngspice on a deck fragment.",
            "inputSchema": {
                "type": "object",
                "required": ["netlist"],
                "properties": {
                    "netlist": {"type": "string"},
                    "analysis": {"type": "string", "enum": ["dc", "ac", "tran"], "default": "tran"},
                    "stop_time": {"type": "number", "default": 5e-8},
                },
            },
        },
        "get_device_parameters": {
            "description": "Fetch geometry + parasitic estimates for a device instance.",
            "inputSchema": {
                "type": "object",
                "required": ["instance"],
                "properties": {
                    "instance": {"type": "string"},
                    "corner": {"type": "string", "default": "tt"},
                },
            },
        },
        "extract_metrics": {
            "description": "Extract metrics from a completed simulation run_id.",
            "inputSchema": {
                "type": "object",
                "required": ["run_id"],
                "properties": {
                    "run_id": {"type": "string"},
                    "metrics": {
                        "type": "array",
                        "items": {"type": "string"},
                        "default": ["dc_gain", "ugf_hz", "tau"],
                    },
                },
            },
        },
    }

    RESOURCE_DEFS = [
        {"uri": "mcp://pdk/models", "name": "pdk_models", "mimeType": "text/plain"},
        {"uri": "mcp://templates/ota_cs", "name": "design_templates", "mimeType": "text/plain"},
    ]

    PROMPT_DEFS = {
        "analog_design_prompt": {
            "description": "Structured prompt for analog block design with margins.",
            "arguments": [
                {"name": "block", "description": "Block name", "required": True},
                {"name": "spec_json", "description": "JSON targets", "required": True},
            ],
        },
        "debug_prompt": {
            "description": "Guided debug tree for suspicious transient behavior.",
            "arguments": [
                {"name": "node", "description": "Probe node", "required": True},
                {"name": "symptom", "description": "Human symptom", "required": True},
            ],
        },
    }

    @app.post("/rpc")
    async def rpc_endpoint(req: Request) -> JSONResponse:
        try:
            body = await req.json()
            msg = JsonRpcReq(**body)
        except Exception as e:
            return JSONResponse(
                {"jsonrpc": "2.0", "error": _err(-32700, "Parse error", str(e)), "id": None},
                status_code=400,
            )

        mid = msg.id
        params = msg.params or {}

        def ok(result: Any) -> Dict[str, Any]:
            return {"jsonrpc": "2.0", "result": result, "id": mid}

        def fail(code: int, message: str, data: Any = None) -> Dict[str, Any]:
            return {"jsonrpc": "2.0", "error": _err(code, message, data), "id": mid}

        method = msg.method

        if method == "initialize":
            return JSONResponse(
                ok(
                    {
                        "protocolVersion": "2024-11-05",
                        "capabilities": {"tools": {}, "resources": {"subscribe": True}, "prompts": {}},
                        "serverInfo": {"name": "ngspice-mcp-mock", "version": "0.1.0"},
                    }
                )
            )

        if method == "tools/list":
            tools = [{"name": n, **meta} for n, meta in TOOL_DEFS.items()]
            return JSONResponse(ok({"tools": tools}))

        if method == "tools/call":
            name = params.get("name")
            args = params.get("arguments") or {}
            if name == "run_simulation":
                netlist = args.get("netlist", "")
                analysis = args.get("analysis", "tran")
                stop_t = float(args.get("stop_time", 5e-8))
                rid = str(uuid.uuid4())
                STATE.publish(rid, {"type": "queued", "analysis": analysis})
                res = _mock_spice(netlist, analysis, stop_t)
                STATE.runs[rid] = res
                STATE.publish(rid, {"type": "completed", "samples": len(res["t"])})
                return JSONResponse(
                    ok({"content": [{"type": "text", "text": json.dumps({"run_id": rid, "preview": res["t_90"]})}]})
                )
            if name == "get_device_parameters":
                inst = str(args.get("instance", "M1"))
                corner = str(args.get("corner", "tt"))
                w = 2.0 + (hash(inst) % 5) * 0.1
                l = 0.03 + (hash(corner) % 3) * 0.005
                payload = {"instance": inst, "corner": corner, "W_um": w, "L_um": l, "nf": 4, "m": 1}
                return JSONResponse(ok({"content": [{"type": "text", "text": json.dumps(payload)}]}))
            if name == "extract_metrics":
                rid = str(args.get("run_id"))
                metrics = args.get("metrics") or ["dc_gain", "ugf_hz", "tau"]
                run = STATE.runs.get(rid)
                if not run:
                    return JSONResponse(fail(-32004, "Unknown run_id", {"run_id": rid}))
                tau = float(run["tau_s"])
                dc_gain_db = 20 * np.log10(max(1e-6, 1.0 - np.exp(-1)))  # toy
                ugf_hz = 1.0 / (2 * np.pi * tau)
                out = {m: None for m in metrics}
                for m in metrics:
                    if m == "tau":
                        out[m] = tau
                    elif m == "dc_gain":
                        out[m] = dc_gain_db
                    elif m == "ugf_hz":
                        out[m] = ugf_hz
                return JSONResponse(ok({"content": [{"type": "text", "text": json.dumps(out)}]}))
            return JSONResponse(fail(-32601, f"Unknown tool: {name}"))

        if method == "resources/list":
            return JSONResponse(ok({"resources": RESOURCE_DEFS}))

        if method == "resources/read":
            uri = params.get("uri")
            if uri == "mcp://pdk/models":
                text = (
                    "* Pedagogical BSIM fragment (NOT FOR FAB)\n"
                    ".model nmos_n12 nmos (LEVEL=54 VERSION=4.5 ...)\n"
                    "+ TOXE=1.2E-9 U0=0.035 VTH0=0.42"
                )
                return JSONResponse(ok({"contents": [{"uri": uri, "mimeType": "text/plain", "text": text}]}))
            if uri == "mcp://templates/ota_cs":
                text = (
                    "* OTA + CMOS CS bias (template)\n"
                    "VDD vdd 0 1.8\n"
                    "IBIAS ibias 0 DC 20u\n"
                    "XOTA in+ in- vdd 0 out ota_core\n"
                    ".tran 1n 500n\n"
                    ".end"
                )
                return JSONResponse(ok({"contents": [{"uri": uri, "mimeType": "text/plain", "text": text}]}))
            return JSONResponse(fail(-32002, "Unknown resource", {"uri": uri}))

        if method == "resources/subscribe":
            rid = str(params.get("run_id"))
            sub_id = str(uuid.uuid4())
            bucket = {"events": []}
            STATE.subscribers.setdefault(rid, []).append(bucket)
            return JSONResponse(ok({"subscriptionId": sub_id, "run_id": rid}))

        if method == "prompts/list":
            prompts = [{"name": n, **v} for n, v in PROMPT_DEFS.items()]
            return JSONResponse(ok({"prompts": prompts}))

        if method == "prompts/get":
            name = params.get("name")
            args2 = params.get("arguments") or {}
            if name == "analog_design_prompt":
                block = args2.get("block", "OTA")
                spec = args2.get("spec_json", "{}")
                messages = [
                    {"role": "system", "content": "You are an analog architect; enforce margins > 10% on GBW, PM, CMRR."},
                    {
                        "role": "user",
                        "content": f"Design {block} meeting {spec}. Enumerate risks: mismatch, headroom, slewing.",
                    },
                ]
                return JSONResponse(ok({"messages": messages}))
            if name == "debug_prompt":
                node = args2.get("node", "v_out")
                sym = args2.get("symptom", "ringing")
                messages = [
                    {"role": "system", "content": "You debug SPICE transients using small-signal and pole reasoning."},
                    {"role": "user", "content": f"Node {node} shows {sym}. Check: load capacitance, compensation zero, supply bounce."},
                ]
                return JSONResponse(ok({"messages": messages}))
            return JSONResponse(fail(-32601, f"Unknown prompt: {name}"))

        return JSONResponse(fail(-32601, f"Method not found: {method}"))

    return app


APP = build_app()
print("FastAPI MCP app constructed: APP")


### 8.3.1 Request / response schemas (JSON)

**`initialize` request**

```json
{
  "jsonrpc": "2.0",
  "method": "initialize",
  "params": {
    "protocolVersion": "2024-11-05",
    "capabilities": {},
    "clientInfo": {"name": "eda-agent", "version": "0.0.1"}
  },
  "id": 1
}
```

**`tools/call` → `run_simulation`**

```json
{
  "jsonrpc": "2.0",
  "method": "tools/call",
  "params": {
    "name": "run_simulation",
    "arguments": {
      "netlist": "* mock RC\\nVIN in 0 PWL(0 0 1n 0 2n 1)\\nR1 in out 1k\\nC1 out 0 100f",
      "analysis": "tran",
      "stop_time": 5e-8
    }
  },
  "id": 42
}
```

**`resources/read` response (excerpt)**

```json
{
  "jsonrpc": "2.0",
  "result": {
    "contents": [
      {"uri": "mcp://pdk/models", "mimeType": "text/plain", "text": ".model nmos_n12 nmos (...)"}
    ]
  },
  "id": 7
}
```

Errors follow JSON‑RPC convention, e.g. `{"code": -32004, "message": "Unknown run_id", "data": {"run_id": "..."}}`.

---


In [ ]:
# Exercise the server in-process with Starlette's TestClient (no open port)
from fastapi.testclient import TestClient

client = TestClient(APP)


def rpc(payload: Dict[str, Any]) -> Dict[str, Any]:
    r = client.post("/rpc", json=payload)
    r.raise_for_status()
    return r.json()


init = rpc(
    {
        "jsonrpc": "2.0",
        "method": "initialize",
        "params": {"protocolVersion": "2024-11-05", "clientInfo": {"name": "notebook", "version": "1"}},
        "id": 1,
    }
)
print("initialize:", json.dumps(init, indent=2)[:800], "...")

tools = rpc({"jsonrpc": "2.0", "method": "tools/list", "params": {}, "id": 2})
print("\nFirst tool:", tools["result"]["tools"][0]["name"])

deck = "* mock RC ladder\\nVIN in 0 PWL(0 0 1n 0 2n 1)\\nR1 in out 1k\\nC1 out 0 100f"
call = rpc(
    {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {"name": "run_simulation", "arguments": {"netlist": deck, "analysis": "tran", "stop_time": 3e-8}},
        "id": 3,
    }
)
run_blob = json.loads(call["result"]["content"][0]["text"])
run_id = run_blob["run_id"]
print("\\nrun_simulation -> run_id:", run_id)

metrics = rpc(
    {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "extract_metrics",
            "arguments": {"run_id": run_id, "metrics": ["tau", "ugf_hz", "dc_gain"]},
        },
        "id": 4,
    }
)
print("extract_metrics:", metrics["result"]["content"][0]["text"])

res_read = rpc({"jsonrpc": "2.0", "method": "resources/read", "params": {"uri": "mcp://pdk/models"}, "id": 5})
print("\\nPDK fragment lines:", len(res_read["result"]["contents"][0]["text"].splitlines()))

prompt = rpc(
    {
        "jsonrpc": "2.0",
        "method": "prompts/get",
        "params": {
            "name": "debug_prompt",
            "arguments": {"node": "v_out", "symptom": "high-frequency ringing"},
        },
        "id": 6,
    }
)
print("\\ndebug_prompt messages:", prompt["result"]["messages"])


## 8.4 MCP client for agent integration

Agents treat MCP as a **tool provider service**:

1. **Discovery** — On connect, call `tools/list` and cache `(name, inputSchema)` in a registry keyed by server id.
2. **Schema validation** — Before `tools/call`, validate arguments with **JSON Schema** (here: `jsonschema` if available; else **Pydantic `TypeAdapter` against a dict skeleton**). We implement a **lightweight required-field checker** to keep the notebook dependency-light.
3. **Retries** — Transient transport failures and **HTTP 429/503** from EDA gateways justify exponential backoff with **jitter**; *not* all JSON‑RPC errors are retryable (`-32602` invalid params should fail fast).
4. **Idempotency** — Long `tools/call` operations (corner batches) should accept an **`Idempotency-Key`** header (or param) so **retries** do not enqueue duplicate SPICE jobs.
5. **Lifecycle** — `async with McpSession(...)` pattern: initialize, optionally `resources/subscribe`, then cancel in-flight on exit.

The implementation below wraps our in-process `TestClient`; swapping `post(base_url + "/rpc", ...)` attaches the same logic to remote farms.

---


In [ ]:
import random
import time
from typing import Any, Dict, List, Optional, Union

from fastapi import FastAPI
from fastapi.testclient import TestClient

JsonRpcId = Union[str, int, None]


class McpSession:
    # Minimal synchronous MCP client with validation + retries (pedagogical).

    def __init__(self, app: FastAPI) -> None:
        self._client = TestClient(app)
        self._initialized = False

    def close(self) -> None:
        self._client.close()

    def __enter__(self) -> "McpSession":
        self.call(
            "initialize",
            {"protocolVersion": "2024-11-05", "clientInfo": {"name": "agent", "version": "0.1"}},
            expect_id=0,
        )
        self._initialized = True
        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        self.close()

    def call(
        self,
        method: str,
        params: Optional[Dict[str, Any]] = None,
        expect_id: int = 1,
        rpc_id: Optional[JsonRpcId] = None,
        idempotency_key: Optional[str] = None,
    ) -> Any:
        rid = rpc_id if rpc_id is not None else expect_id
        payload = {"jsonrpc": "2.0", "method": method, "params": params or {}, "id": rid}
        headers = {}
        if idempotency_key:
            headers["Idempotency-Key"] = idempotency_key  # HTTP gateways dedupe replayed tool calls
        r = self._client.post("/rpc", json=payload, headers=headers)
        r.raise_for_status()
        msg = r.json()
        if msg.get("error"):
            raise RuntimeError(msg["error"])
        return msg["result"]

    @staticmethod
    def _check_required(schema: Dict[str, Any], args: Dict[str, Any]) -> None:
        req = schema.get("required") or []
        for k in req:
            if k not in args:
                raise ValueError(f"Missing required field '{k}' for tool schema {schema}")

    def tools(self) -> List[Dict[str, Any]]:
        return self.call("tools/list", {}, expect_id=10)["tools"]

    def call_tool(self, name: str, arguments: Dict[str, Any], max_retries: int = 4) -> Any:
        registry = {t["name"]: t for t in self.tools()}
        if name not in registry:
            raise KeyError(f"Unknown tool {name}")
        schema = registry[name]["inputSchema"]
        self._check_required(schema, arguments)

        delay = 0.05
        last_err: Optional[Exception] = None
        for attempt in range(max_retries):
            try:
                return self.call(
                    "tools/call",
                    {"name": name, "arguments": arguments},
                    expect_id=11 + attempt,
                )
            except Exception as e:  # noqa: BLE001 - pedagogical catch
                last_err = e
                # Retry only on synthetic transport flakiness we inject below (here: always fail fast)
                time.sleep(delay + random.random() * delay)
                delay *= 2
        raise RuntimeError(last_err)


with McpSession(APP) as sess:
    tool_names = [t["name"] for t in sess.tools()]
    print("Discovered tools:", tool_names)
    dev = sess.call_tool("get_device_parameters", {"instance": "XM13", "corner": "ss"})
    print("get_device_parameters:", dev)

print("Session closed cleanly.")


## 8.5 Advanced MCP patterns

### 8.5.1 Chaining multiple MCP servers (Ngspice + ALIGN + OpenROAD)

**Pattern:** A **federation proxy** prefixes tool names (`ngspice.*`, `align.*`, `openroad.*`) so the LLM sees a **flat** tool namespace while each backend keeps isolated credentials. Conflicts in JSON Schema are resolved by **version suffixing** (`run_simulation@v2`).

### 8.5.2 Resource subscriptions for simulation progress

Long **corner sweeps** expose **percent complete**, **current netlist hash**, and **ETA**. Servers push **notifications** (JSON‑RPC notifications without `id`) or, on HTTP, **SSE chunks**. Our toy server stores `STATE.subscribers[run_id]`.

### 8.5.3 Prompt templates for domain reasoning

Use **Jinja2** (already in course requirements) to compose **prompt packets** from measured metrics + PDK excerpts, ensuring the **model** receives *structured* context instead of raw log dumps.

---


In [ ]:
import json
from typing import Any, Dict, Iterable, List

from jinja2 import Template

# --- Federated namespace (three toy backends in one dict) ---


class ToyBackend:
    # Named tool group (one MCP server advertisement).

    def __init__(self, tools: List[Dict[str, Any]]) -> None:
        self._tools = tools

    def tools(self) -> List[Dict[str, Any]]:
        return self._tools


NG_TOOLS = [
    {
        "name": "ngspice.run_ac",
        "description": "Run AC analysis (toy stand-in for corner sweep).",
        "inputSchema": {"type": "object", "required": ["fstart_hz"], "properties": {"fstart_hz": {"type": "number"}}},
    },
    {
        "name": "ngspice.run_noise",
        "description": "Run noise summary (toy).",
        "inputSchema": {"type": "object", "required": ["src"], "properties": {"src": {"type": "string"}}},
    },
]
ALIGN_TOOLS = [
    {
        "name": "align.place_macro",
        "description": "Place analog macro in ALIGN canvas (toy).",
        "inputSchema": {"type": "object", "required": ["macro"], "properties": {"macro": {"type": "string"}}},
    },
    {
        "name": "align.route_power",
        "description": "Route power mesh stub (toy).",
        "inputSchema": {"type": "object", "required": ["net"], "properties": {"net": {"type": "string"}}},
    },
]
OR_TOOLS = [
    {
        "name": "openroad.global_route",
        "description": "Global routing in OpenROAD (toy).",
        "inputSchema": {"type": "object", "required": ["clk_net"], "properties": {"clk_net": {"type": "string"}}},
    },
    {
        "name": "openroad.timing_drv",
        "description": "Report timing DRV (toy).",
        "inputSchema": {"type": "object", "required": ["corner"], "properties": {"corner": {"type": "string"}}},
    },
]


def merge_tool_catalog(backends: Iterable[ToyBackend]) -> Dict[str, Dict[str, Any]]:
    flat: Dict[str, Dict[str, Any]] = {}
    for b in backends:
        for t in b.tools():
            if t["name"] in flat:
                raise ValueError(f"Collision on {t['name']}")
            flat[t["name"]] = t
    return flat


catalog = merge_tool_catalog([ToyBackend(NG_TOOLS), ToyBackend(ALIGN_TOOLS), ToyBackend(OR_TOOLS)])
print("Federated tools:", sorted(catalog.keys()))

# Subscription demo: create run, attach subscriber, replay events
STATE2 = NgspiceMcpState()
STATE2.subscribers.setdefault("runX", []).append({"events": []})
STATE2.publish("runX", {"pct": 10, "stage": "netlist_parse"})
STATE2.publish("runX", {"pct": 55, "stage": "tran_step"})
STATE2.publish("runX", {"pct": 100, "stage": "done"})
print("Subscription trace:", STATE2.subscribers["runX"][0]["events"])

# Jinja prompt template combining metrics + resource snippet
_tmpl_src = (
    "You are reviewing an analog block {{ block }}.\n"
    "Metrics: {{ metrics_json }}\n"
    "PDK note: {{ pdk_snippet[:200] }}...\n"
    "Task: propose *two* design changes ranked by expected GBW impact."
)
tmpl = Template(_tmpl_src)
pdk_snippet = "TOXE=1.2E-9 U0=0.035 ... (synthetic)"
prompt_text = tmpl.render(
    block="OTA",
    metrics_json=json.dumps({"PM_deg": 62, "GBW_MHz": 120}),
    pdk_snippet=pdk_snippet,
)
print("\\nRendered prompt:\\n", prompt_text)

# Plotly: synthetic retry latency distribution (toy)
rng = np.random.default_rng(3)
attempts = np.arange(1, 6)
lat_ms = np.cumsum(rng.lognormal(mean=2.5, sigma=0.35, size=len(attempts)))

fig = go.Figure(
    go.Scatter(
        x=attempts,
        y=lat_ms,
        mode="lines+markers",
        line=dict(color=ACCENT, width=2),
        marker=dict(size=10, color=GREEN),
        name="Backoff path",
    )
)
fig.update_layout(
    title="Toy: cumulative client retry latency vs attempt (lognormal jitter)",
    xaxis_title="Attempt index",
    yaxis_title="Cumulative latency (ms)",
    paper_bgcolor=DARK_BG,
    plot_bgcolor=PANEL,
    yaxis=dict(gridcolor="#21262d"),
    xaxis=dict(gridcolor="#21262d"),
    height=400,
)
fig.show()


## 8.6 Takeaways

- MCP **separates concerns**: hosts enforce **policy**, clients manage **sessions**, servers encapsulate **dangerous or valuable** EDA state.
- For analog, **tools** map naturally to **SPICE/LVS/DRC** invocations; **resources** encapsulate **PDK fragments**; **prompts** standardize **debug playbooks**.
- A **FastAPI** POST `/rpc` surface is an excellent teaching stand‑in for **stdio/SSE** transports because the **JSON‑RPC payloads are identical**.
- Production hardening adds **OAuth2/mTLS**, **per‑tool rate limits**, **audit logs** of `tools/call`, and **content addressing** for resources.

### Suggested exercises

1. Replace `_mock_spice` with a **subprocess** call to Ngspice and stream stdout lines via **SSE**.
2. Implement **JSON Schema** validation with `jsonschema` for every `tools/call`.
3. Add **`$/progress` notifications** during multi-corner sweeps and graph them live in Plotly.

---
